In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')


In [ ]:
import collections, gc, hashlib, json, math, os, random, re, shutil, sys, time
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from safetensors import safe_open
from safetensors.torch import save_file, load_file
from accelerate import init_empty_weights
from transformers import AutoTokenizer, Gemma4TextConfig, Gemma4ForCausalLM
from compressed_tensors.compressors.pack_quantized import PackedQuantizationCompressor
from compressed_tensors.quantization import QuantizationScheme
import peft.import_utils as _peft_iu
import peft.tuners.lora.torchao as _peft_tao
_peft_iu.is_torchao_available = lambda: False
_peft_tao.is_torchao_available = lambda: False
from peft import LoraConfig, get_peft_model
from transformers.integrations.sdpa_attention import sdpa_attention_forward
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS

# ================= LORA V2 CONFIG =================
DECLARED_MODEL = 'gemma-4-e4b-it-qat-w4a16-ct'
MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{DECLARED_MODEL}/2')
KIT_CANDIDATES = [
    Path('/kaggle/input/datasets/dmitriigluzdov/gemma-4-serving-and-lora-verification-kit'),
    Path('/kaggle/input/gemma-4-serving-and-lora-verification-kit'),
]
KIT_DIR = next((p for p in KIT_CANDIDATES if p.is_dir()), None)
assert MODEL_PATH.is_dir() and KIT_DIR is not None

WORKING_DIR = Path('/kaggle/working')
OUT_DIR = WORKING_DIR / 'lora_v2_run'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_MID_DIR = OUT_DIR / 'adapters' / 'e4b_lora_v2_mid'
ADAPTER_FINAL_DIR = OUT_DIR / 'adapters' / 'e4b_lora_v2_final'

STRIP_THOUGHTS = True
MAX_CTX = 8192
MAX_TRAIN_STEPS = 1280     # 160 optimizer updates at grad_accum=8
MAX_VAL_STEPS = 64
LORA_R = 32                # 2x rank capacity (~134 MB)
LORA_ALPHA = 16            # scale = 0.5 (strictly <= 1.0)
LORA_DROPOUT = 0.05
LR = 2e-5
GRAD_ACCUM = 8
SEED = 1337

SOUND24_IDS = set(["fastapi_13537", "fastapi_13786", "fastapi_14349", "fastapi_14360", "fastapi_14487", "fastapi_14512", "fastapi_14873", "fastapi_14964", "fastapi_14978", "fastapi_15588", "fastapi_15763", "httpx_3672", "requests_6644", "requests_7315", "requests_7427", "requests_7505", "rich_3063", "rich_3468", "rich_3469", "rich_3506", "rich_3518", "rich_3676", "rich_3782", "rich_4077"])
BANNED_TOOLS = ('search_similar_code', 'get_code_neighbors', 'get_code_subgraph')
THOUGHT_RE = re.compile(r'<\|channel>thought\n.*?<channel\|>', re.DOTALL)

torch.manual_seed(SEED)
random.seed(SEED)
sys.path.insert(0, str(KIT_DIR))
from lora_data import iter_trajectories

tok = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

def clean_thoughts(text: str) -> str:
    return THOUGHT_RE.sub('<|channel>thought\n<channel|>', text)

def classify_action(comp: str) -> str:
    if 'submit_patch' in comp:
        return 'submit'
    if 'edit_file' in comp or 'write_file' in comp:
        return 'edit'
    if 'run_command' in comp:
        return 'cmd'
    if 'read_file' in comp:
        return 'read'
    return 'other'

print('Scanning ALL 1,670 trajectories for complete resolved episodes (post-strip <= 8192)...', flush=True)
trajs_kept = 0
buckets = collections.defaultdict(list)
val_pool = []

for tier in ('native', 'external_A', 'external_B', 'external_B2'):
    for rec in iter_trajectories(KIT_DIR, tiers=(tier,), max_attempts_per_issue=2):
        issue = str(rec.get('_issue') or '')
        if issue in SOUND24_IDS:
            continue
        exs = rec.get('sft_examples') or []
        if not exs:
            continue
        submit_idx = next((i for i, e in enumerate(exs) if 'submit_patch' in (e.get('completion') or '')), None)
        if submit_idx is None:
            continue
        ep = exs[:submit_idx + 1]
        if max(e.get('context_tokens', 999999) for e in ep) > 20000:
            continue
        if any(any(bt in (e.get('completion') or '') for bt in BANNED_TOOLS) for e in ep):
            continue

        # Check post-stripped final submit step fits MAX_CTX
        last_e = ep[-1]
        lp = clean_thoughts(last_e['prefix'])
        lc = clean_thoughts(last_e['completion'])
        if len(tok.encode(lp, add_special_tokens=False)) + len(tok.encode(lc, add_special_tokens=False)) > MAX_CTX:
            continue

        ep_items = []
        valid_ep = True
        for e in ep:
            prompt_txt = clean_thoughts(e['prefix'])
            comp_txt = clean_thoughts(e['completion'])
            p_ids = tok.encode(prompt_txt, add_special_tokens=False)
            c_ids = tok.encode(comp_txt, add_special_tokens=False)
            if len(p_ids) < 2 or len(c_ids) < 1 or (len(p_ids) + len(c_ids)) > MAX_CTX:
                valid_ep = False
                break
            ep_items.append({
                'input_ids': p_ids + c_ids,
                'prompt_tokens': len(p_ids),
                'target_tokens': len(c_ids),
                'context_tokens': len(p_ids) + len(c_ids),
                'tier': tier,
                'action': classify_action(comp_txt),
                'issue': issue,
            })
        if not valid_ep or not ep_items:
            continue

        trajs_kept += 1
        is_val = (trajs_kept % 12 == 0)
        for item in ep_items:
            if is_val:
                val_pool.append(item)
            else:
                buckets[item['action']].append(item)

rng = random.Random(SEED)
for k in buckets:
    rng.shuffle(buckets[k])
rng.shuffle(val_pool)

print(f'Complete episodes kept across ALL tiers: {trajs_kept}', flush=True)
print('Raw action counts:', {k: len(v) for k, v in buckets.items()}, flush=True)

quotas = {
    'edit': int(MAX_TRAIN_STEPS * 0.36),
    'submit': int(MAX_TRAIN_STEPS * 0.20),
    'cmd': int(MAX_TRAIN_STEPS * 0.26),
    'read': int(MAX_TRAIN_STEPS * 0.16),
    'other': int(MAX_TRAIN_STEPS * 0.02),
}
train_examples = []
for act, q in quotas.items():
    pool = buckets.get(act, [])
    if not pool:
        continue
    if len(pool) >= q:
        train_examples.extend(pool[:q])
    else:
        mult = 2 if act in ('edit', 'submit') else 1
        train_examples.extend((pool * mult)[:q])

# If unique steps < 960, run 2 clean epochs so we get enough optimizer updates
if len(train_examples) < 960:
    ep2 = list(train_examples)
    rng.shuffle(ep2)
    train_examples = (train_examples + ep2)[:MAX_TRAIN_STEPS]

rng.shuffle(train_examples)
val_examples = val_pool[:MAX_VAL_STEPS]
print('Selected train:', len(train_examples), collections.Counter(x['action'] for x in train_examples), flush=True)
print('Selected val:', len(val_examples), flush=True)

def chunked_sdpa(module, q, k, v, attention_mask, dropout=0., scaling=None, **kw):
    if q.shape[-1] <= 256:
        return sdpa_attention_forward(module, q, k, v, attention_mask, dropout=dropout, scaling=scaling, **kw)
    assert dropout == 0 and q.shape[-2] == k.shape[-2] and q.shape[0] == 1
    if k.shape[1] != q.shape[1]:
        factor = q.shape[1] // k.shape[1]
        k = k.repeat_interleave(factor, 1)
        v = v.repeat_interleave(factor, 1)
    outs = []
    L = q.shape[-2]
    for lo in range(0, L, 128):
        hi = min(lo + 128, L)
        mask = attention_mask[:, :, lo:hi, :L] if attention_mask is not None else (
            torch.arange(L, device=q.device)[None, None, None, :] <= torch.arange(lo, hi, device=q.device)[None, None, :, None]
        )
        def attend(x, y, z, m):
            return F.scaled_dot_product_attention(x, y, z, attn_mask=m, dropout_p=0., is_causal=False, scale=scaling)
        outs.append(checkpoint(attend, q[:, :, lo:hi], k, v, mask, use_reentrant=False) if torch.is_grad_enabled() else attend(q[:, :, lo:hi], k, v, mask))
    return torch.cat(outs, dim=2).transpose(1, 2).contiguous(), None

ALL_ATTENTION_FUNCTIONS.register('sdpa', chunked_sdpa)

def load_exact_e4b_base(model_dir: Path, device: str = 'cuda:0'):
    raw = json.loads((model_dir / 'config.json').read_text())
    cfg = Gemma4TextConfig(**raw['text_config'])
    cfg._attn_implementation = 'sdpa'
    cfg.use_cache = False
    with init_empty_weights():
        base = Gemma4ForCausalLM(cfg)
    groups = raw['quantization_config']['config_groups']
    schemes = [QuantizationScheme(**g) for g in groups.values()]
    assert len(schemes) == 1
    with safe_open(model_dir / 'model.safetensors', framework='pt', device='cpu') as f:
        keys = set(f.keys())
        for name, p in list(base.named_parameters()):
            if name == 'lm_head.weight':
                continue
            source = 'model.language_model.' + name.removeprefix('model.')
            parent, attr = name.rsplit('.', 1)
            if source in keys:
                w = f.get_tensor(source)
            else:
                stem = source.removesuffix('.weight')
                state = {n.rsplit('.', 1)[-1]: f.get_tensor(n) for n in keys if n.startswith(stem + '.')}
                w = PackedQuantizationCompressor.decompress(state, schemes[0])['weight']
            setattr(base.get_submodule(parent), attr, nn.Parameter(w.to(device), requires_grad=False))
        base.lm_head.weight = base.model.embed_tokens.weight
    buf_count = 0
    with safe_open(model_dir / 'model.safetensors', framework='pt', device='cpu') as f:
        keys = set(f.keys())
        for name, buf in list(base.named_buffers()):
            source = 'model.language_model.' + name.removeprefix('model.')
            parent, attr = name.rsplit('.', 1)
            if source in keys:
                value = f.get_tensor(source)
                buf_count += 1
            else:
                value = buf
            setattr(base.get_submodule(parent), attr, value.to(device))
    assert buf_count == 42
    base.requires_grad_(False)
    return base

def compute_loss(base, ex, backward=False, denom=1, device='cuda:0'):
    ids = torch.tensor([ex['input_ids']], device=device)
    p = ex['prompt_tokens']
    target = ids[0, p:]
    h = base.model(input_ids=ids, use_cache=False).last_hidden_state[0, p - 1:-1]
    total = 0.0
    def ce(x, y):
        logits = F.linear(x, base.lm_head.weight)
        logits = 30.0 * torch.tanh(logits / 30.0)
        return F.cross_entropy(logits.float(), y, reduction='sum')
    for i in range(0, len(target), 32):
        z = checkpoint(ce, h[i:i + 32], target[i:i + 32], use_reentrant=False) if backward else ce(h[i:i + 32], target[i:i + 32])
        total = total + z
    result = total / len(target)
    if backward:
        (result / denom).backward()
    return float(result.detach())

def save_vllm_adapter(model, path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path, safe_serialization=True)
    tensors = load_file(path / 'adapter_model.safetensors')
    mapped = {
        n.replace('base_model.model.model.layers.', 'base_model.model.model.language_model.layers.'): t.to(torch.bfloat16).contiguous()
        for n, t in tensors.items()
    }
    assert len(mapped) == 516
    save_file(mapped, path / 'adapter_model.safetensors')
    cfg = json.loads((path / 'adapter_config.json').read_text())
    cfg.update(base_model_name_or_path='google/gemma-4-E4B-it-qat-w4a16-ct', task_type='CAUSAL_LM', inference_mode=True)
    cfg['target_modules'] = [n.replace('model.layers.', 'model.language_model.layers.') for n in cfg['target_modules']]
    (path / 'adapter_config.json').write_text(json.dumps(cfg, indent=2) + '\n')
    print(f'Saved vLLM adapter to {path} ({(path / "adapter_model.safetensors").stat().st_size / 1e6:.1f} MB)', flush=True)

t0 = time.time()
base = load_exact_e4b_base(MODEL_PATH, device='cuda:0')
targets = [n for n, _ in base.named_modules() if re.fullmatch(r'model\.layers\.\d+\.(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)', n)]
assert len(targets) == 258
base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
base.enable_input_require_grads()
model = get_peft_model(base, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM', target_modules=targets,
))
params = [p for _, p in model.named_parameters() if p.requires_grad]
print(f'LoRA v2 trainable params: {sum(p.numel() for p in params):,} (r={LORA_R}, alpha={LORA_ALPHA}, scale={LORA_ALPHA/LORA_R})', flush=True)

optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0, foreach=False)

def eval_val():
    if not val_examples:
        return None
    model.eval()
    with torch.no_grad():
        items = [(compute_loss(base, e), e['target_tokens']) for e in val_examples]
    model.train()
    return sum(l * n for l, n in items) / sum(n for _, n in items)

val_before = eval_val()
print(f'Val NLL before: {val_before}', flush=True)

updates = math.ceil(len(train_examples) / GRAD_ACCUM)
warmup = max(1, math.ceil(updates * 0.05))
mid_step = max(1, updates // 2)
model.train()
curve = []
val_mid = None
for step, start in enumerate(range(0, len(train_examples), GRAD_ACCUM), 1):
    group = train_examples[start:start + GRAD_ACCUM]
    lr_now = LR * (step / warmup if step <= warmup else 0.5 * (1.0 + math.cos(math.pi * (step - warmup) / max(1, updates - warmup))))
    for g in optimizer.param_groups:
        g['lr'] = lr_now
    optimizer.zero_grad(set_to_none=True)
    tick = time.time()
    losses = [compute_loss(base, ex, backward=True, denom=len(group)) for ex in group]
    assert all(math.isfinite(z) for z in losses)
    gnorm = float(torch.nn.utils.clip_grad_norm_(params, 1.0))
    optimizer.step()
    row = dict(step=step, updates=updates, lr=round(lr_now, 8), loss=round(sum(losses)/len(losses), 4), gnorm=round(gnorm, 3), sec=round(time.time()-tick, 1))
    curve.append(row)
    if step == 1 or step % 15 == 0 or step == updates:
        print(json.dumps(row), flush=True)
    if step == mid_step:
        val_mid = eval_val()
        print(f'Val NLL mid (step {step}): {val_mid}', flush=True)
        save_vllm_adapter(model, ADAPTER_MID_DIR)

val_after = eval_val()
print(f'Val NLL after (step {updates}): {val_after} (elapsed {(time.time()-t0)/60:.1f} min)', flush=True)
save_vllm_adapter(model, ADAPTER_FINAL_DIR)
(OUT_DIR / 'train_curve.json').write_text(json.dumps(dict(trajs_kept=trajs_kept, val_before=val_before, val_mid=val_mid, val_after=val_after, curve=curve), indent=2))

del model, base, optimizer, params
gc.collect()
torch.cuda.empty_cache()
print('GPU freed cleanly for 24-task vLLM evaluation.', flush=True)


In [ ]:
import asyncio, concurrent.futures, zipfile
import litellm, pandas as pd, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import EvalConfig, build_submission_limits, ALLOWED_ADAPTER_EXTENSIONS
from swegemma.evaluate import Evaluator
from swegemma.models import load_tasks

litellm.drop_params = True
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
BASE_V2C = {
  "agent.yaml": "agent_class: SequentialAgent\nname: v2c_seq2_t0_t0_agent\ndescription: Two-stage fresh-context SWE repair agent (Stage 1 coder without submit_patch -> Stage 2 adversarial verifier/fixer without write_file).\nsub_agents:\n  - config_path: sub_agents/coder.yaml\n  - config_path: sub_agents/verifier.yaml\n",
  "configs/sampling_s1.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  thinking_budget: 0\n  include_thoughts: false\n",
  "configs/sampling_s2.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  thinking_budget: 0\n  include_thoughts: false\n",
  "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 36\n  max_time_minutes: 8\n  max_turns: 80\n",
  "prompts/coder.md": "You are Stage 1 (Coder) in `/workspace`.\nYour ONLY available tools are: `run_command`, `read_file`, `edit_file`, `write_file`, `get_status`. Always invoke shell commands through `run_command(command=...)`.\nGoal: quickly locate the exact source file and apply a clean initial fix in ~10\u201314 tool calls, then hand off to Stage 2.\n\n## Steps\n1. **Locate (calls 1\u20135)**:\n   - Call `run_command` with a targeted search such as `git grep -n -F -e 'SymbolName' -- '*.py' | head -25`.\n   - Call `read_file` on the relevant 80\u2013120 line slice (never dump huge files).\n2. **Edit source code (by call 10)**:\n   - Call `edit_file` with a short, unique `old_string` and `allow_multiple=false`.\n   - For FastAPI doc-code issues, edit executable files under `docs_src/`.\n   - If `edit_file` fails twice, call `run_command` with a short inline Python exact-replace script and compile check.\n   - NEVER call `write_file` to create test or scratch scripts (never pass `/tmp/...` to `write_file`\u2014`write_file` writes inside `/workspace`).\n3. **Compile & hand off (calls 11\u201314)**:\n   - Call `run_command` with `python3 -m py_compile <edited_file>` and optionally one short inline `python3 -c \"...\"` check.\n   - Stop as soon as the edit compiles and output a 4-line handoff summary: edited files, root cause, change made, and edge cases to check.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER run bare `pytest` or `pip install`.\n",
  "prompts/verifier.md": "You are Stage 2 (Adversarial Verifier & Fixer) with a fresh context in `/workspace`.\nYour ONLY available tools are: `run_command`, `read_file`, `edit_file`, `get_status`, `submit_patch`. Always invoke shell commands through `run_command(command=...)`.\nStage 1 often edits the right file with a subtle bug, missed edge case, wrong exception/message, or incomplete logic. Audit critically, fix any flaw, clean stray files, and call `submit_patch`.\n\n## Problem Statement\n{problem_description}\n\n## Stage 1 Handoff\n{coder_notes?}\n\n## Steps\n1. **Audit current diff (call 1)**:\n   - Call `run_command` with `git status -s && git diff`.\n   - If the diff is empty, locate the source file immediately (`run_command` with `git grep`) and implement the fix with `edit_file`.\n   - If any untracked scratch files (`tmp/`, `verify_fix.py`, `repro.py`, `test_*.py`) or edits to protected test/config files exist in `/workspace`, clean/revert them via `run_command` (`rm -rf` / `git checkout --`).\n2. **Inspect full context (calls 2\u20134)**:\n   - Call `read_file` on 80\u2013120 lines around every edited function.\n   - Verify EVERY requirement in the problem statement: exact return value/type, exception type & message, default args, None/empty/boundary edge cases, and sibling callers.\n3. **Run focused verification (calls 5\u20138)**:\n   - Call `run_command` to run `python3 -m py_compile <edited_files>` AND a short inline `python3 -c \"...\"` script testing the reported bug + 2 edge cases + 1 normal control case.\n   - You may also run ONE targeted pytest node (`pytest tests/path/test_file.py -k <name>`) via `run_command`. If `pytest` is disabled, do NOT retry it\u2014use inline `python3 -c` assertions.\n   - NEVER write test files inside `/workspace` and NEVER loop more than twice debugging your own check script.\n4. **Repair if needed & submit**:\n   - If your audit or check reveals ANY defect, fix the source file with `edit_file` and re-run `py_compile` + inline check.\n   - Verify `git status -s` shows ONLY intended source edits (no stray files, no test/config changes).\n   - Call `submit_patch` as your final tool action while budget remains, confirm `patch_size > 0` and `files_changed > 0`, then give a 1-sentence summary.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER run bare `pytest`, `pip install`, or `git commit`.\n- NEVER finish without calling `submit_patch` on a compiled, non-empty source fix.\n",
  "sub_agents/coder.yaml": "name: coder_agent\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Stage 1 coder that locates the root cause and applies an initial source fix in /workspace without submitting.\ninstruction: !include ../prompts/coder.md\noutput_key: coder_notes\ndisallow_transfer_to_parent: true\ndisallow_transfer_to_peers: true\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\ngenerate_content_config: !include ../configs/sampling_s1.yaml\n",
  "sub_agents/verifier.yaml": "name: verifier_fixer_agent\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Stage 2 fresh-context adversarial verifier and fixer that audits git diff, runs focused inline checks, repairs bugs, cleans scratch files, and calls submit_patch.\ninstruction: !include ../prompts/verifier.md\ninclude_contents: none\ndisallow_transfer_to_parent: true\ndisallow_transfer_to_peers: true\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include ../configs/sampling_s2.yaml\n"
}

# Build a combined adapters staging dir so ONE vLLM server serves BOTH mid and final LoRA checkpoints!
CHECKPOINTS = {
    'v4a_lora_v2_mid': ('e4b_lora_v2_mid', ADAPTER_MID_DIR),
    'v4b_lora_v2_final': ('e4b_lora_v2_final', ADAPTER_FINAL_DIR),
}

sub_dirs = {}
for run_name, (aname, asrc) in CHECKPOINTS.items():
    sdir = WORKING_DIR / f'sub_{run_name}'
    if sdir.exists():
        shutil.rmtree(sdir)
    for rel, txt in BASE_V2C.items():
        p = sdir / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        if rel.endswith('.yaml') and 'model: gemma-4-e4b-it-qat-w4a16-ct' in txt:
            txt = txt.replace(
                'model: gemma-4-e4b-it-qat-w4a16-ct\n',
                f'model: gemma-4-e4b-it-qat-w4a16-ct\nadapter: {aname}\n',
            )
        p.write_text(txt, encoding='utf-8')
    shutil.copytree(asrc, sdir / 'adapters' / aname)
    zout = WORKING_DIR / f'{run_name}_submission.zip'
    with zipfile.ZipFile(zout, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for fp in sorted(sdir.rglob('*')):
            if fp.is_file():
                zf.write(fp, arcname=str(fp.relative_to(sdir)))
    print(f'Packed {zout} ({zout.stat().st_size / 1e6:.1f} MB)', flush=True)
    sub_dirs[run_name] = sdir

# Staging dir containing BOTH adapters for single vLLM multi-LoRA launch
MULTI_ADAPTER_ROOT = WORKING_DIR / 'multi_adapters_stage'
if MULTI_ADAPTER_ROOT.exists():
    shutil.rmtree(MULTI_ADAPTER_ROOT)
shutil.copytree(ADAPTER_MID_DIR, MULTI_ADAPTER_ROOT / 'adapters' / 'e4b_lora_v2_mid')
shutil.copytree(ADAPTER_FINAL_DIR, MULTI_ADAPTER_ROOT / 'adapters' / 'e4b_lora_v2_final')
adapters = discover_adapters(str(MULTI_ADAPTER_ROOT), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=2,
    max_lora_rank=LORA_R,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=(),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM multi-LoRA server started on {server_instance.base_url} (tp={tp_size}, max_lora_rank={LORA_R})', flush=True)

models = server_instance.create_model_registry(
    aliases=[DECLARED_MODEL],
    model_prefix='openai/',
    api_key=os.environ.get('LOCAL_API_KEY', 'EMPTY'),
)

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if (callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn)) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

limits, gen_constraints = build_submission_limits()
tasks = load_tasks(DATA_DIR / 'tasks.jsonl')
subset = sorted([t for t in tasks if t.instance_id in SOUND24_IDS], key=lambda t: t.instance_id)
assert len(subset) == 24

all_summaries = []
for run_name, sdir in sub_dirs.items():
    print(f'\n================ EVALUATING {run_name} ({len(subset)} sound tasks) ================', flush=True)
    ev_raw = yaml.safe_load((sdir / 'eval_config.yaml').read_text()) or {}
    ev = ev_raw.get('evaluation', ev_raw)
    rdir = OUT_DIR / f'eval_{run_name}'
    eval_config = EvalConfig(
        tasks_path=DATA_DIR / 'tasks.jsonl',
        snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=rdir,
        submission_dir=sdir,
        models=models,
        sandbox='subprocess',
        timeout_seconds=int(ev.get('timeout_seconds', 240)),
        verification_timeout_seconds=300,
        max_time_minutes=float(ev.get('max_time_minutes', 8.0)),
        max_tool_calls=int(ev.get('max_tool_calls', 36)),
        max_turns=int(ev.get('max_turns', 80)),
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'),
        embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
    )
    evaluator = Evaluator(eval_config)
    rows = []
    for idx, task in enumerate(subset, 1):
        t_s = time.perf_counter()
        try:
            res = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(subset))
            patch = (res.agent_patch or '').strip()
            row = dict(run=run_name, id=task.instance_id, repo=task.repo, resolved=bool(res.resolved),
                       patch_chars=len(patch), tool_calls=res.tool_calls,
                       agent_seconds=round(float(res.duration_seconds or 0.0), 1),
                       test_exit_code=res.test_exit_code, error=getattr(res, 'error', None))
        except Exception as exc:
            row = dict(run=run_name, id=task.instance_id, repo=task.repo, resolved=False, patch_chars=0,
                       tool_calls=None, agent_seconds=None, test_exit_code=None, error=repr(exc)[:300])
        row['wall_seconds'] = round(time.perf_counter() - t_s, 1)
        rows.append(row)
        print(f"[{run_name} {idx}/{len(subset)}] {row['id']}: resolved={row['resolved']} patch={row['patch_chars']} calls={row['tool_calls']} agent={row['agent_seconds']}s err={row['error']}", flush=True)
    df = pd.DataFrame(rows)
    summary = dict(
        run=run_name, tasks=len(df), resolved=int(df['resolved'].sum()),
        resolution_rate=round(float(df['resolved'].mean()), 3),
        no_patch=int((df['patch_chars'] == 0).sum()),
        errors=int(df['error'].notna().sum()),
        mean_tool_calls=round(float(df['tool_calls'].dropna().mean()), 1) if len(df['tool_calls'].dropna()) else None,
        agent_seconds_median=float(df['agent_seconds'].dropna().median()) if len(df['agent_seconds'].dropna()) else None,
        wall_seconds_mean=round(float(df['wall_seconds'].mean()), 1),
    )
    (WORKING_DIR / f'{run_name}_per_task.jsonl').write_text('\n'.join(json.dumps(r) for r in rows) + '\n')
    all_summaries.append(summary)
    print(json.dumps(summary, indent=2), flush=True)
    print(df.to_string(index=False), flush=True)

(WORKING_DIR / 'lora_v2_eval24_summary.json').write_text(json.dumps(all_summaries, indent=2))
print('\n=== FINAL LORA V2 24-TASK COMPARISON ===', flush=True)
print(pd.DataFrame(all_summaries).to_string(index=False), flush=True)
